# RAG Capstone: The Whole Pipeline on One Corpus

Every earlier notebook studied one stage. This one **wires them together** on the course corpus (HotpotQA, indexed in `6.5`) and measures each stage separately, so a wrong answer can be traced to the stage that caused it.

```
question → hybrid retrieve (BM25 + dense, RRF) → cross-encoder rerank → answer with citations → evaluate
              6.7                                      6.9                    6.12 idea             6.11 idea
```

## Overview
1. **Retrieve**: BM25 and dense search, fused with Reciprocal Rank Fusion (6.7).
2. **Rerank**: a cross-encoder keeps the best 3 of 10 candidates (6.9).
3. **Answer with citations**: the LLM answers from numbered passages and names the ones it used (6.12).
4. **Evaluate by stage**: retrieval, citation and answer quality are scored separately, then failures are labelled by the first stage that went wrong.
5. **Compare**: the finished pipeline is added to the shared results ledger next to every earlier technique.

**Prerequisites:** run `6.5` first (builds the index), Qdrant and Ollama running.

---
## 1. Setup

In [ ]:
%pip install -q rank_bm25 langchain-qdrant langchain-huggingface langchain-openai qdrant-client sentence-transformers rapidfuzz

In [ ]:
import json
import re
from statistics import mean

import numpy as np
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

import rag_common as rc

cfg = rc.load_config()
rc.check_services(cfg)
qa_pairs, passages = rc.load_hotpotqa(cfg)
client, embeddings, doc_store = rc.connect_qdrant(cfg)
llm_name, llm = next(iter(rc.load_llms(cfg).items()))

eval_set = rc.eval_set(qa_pairs)   # the same fixed question sample every RAG notebook scores on
print(f"{len(passages)} passages | {len(qa_pairs)} questions | scoring on {len(eval_set)} | LLM: {llm_name}")

---
## 2. Stage 1: Hybrid Retrieval

BM25 catches exact names and rare terms, dense search catches paraphrases. RRF merges the two ranked lists using ranks only (6.7).

In [ ]:
titles = [p["title"] for p in passages]
texts = [p["text"] for p in passages]
text_of = dict(zip(titles, texts))

def tokenize(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([tokenize(t) for t in texts])

def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, item in enumerate(ranking, start=1):
            scores[item] = scores.get(item, 0.0) + 1.0 / (k + rank)
    return [item for item, _ in sorted(scores.items(), key=lambda kv: kv[1], reverse=True)]

def retrieve(question, pool=20, top=10):
    """Hybrid candidates: titles of the top `top` passages after fusing BM25 and dense rankings."""
    lexical = [titles[i] for i in np.argsort(-bm25.get_scores(tokenize(question)))[:pool]]
    dense = [d.metadata["title"] for d in doc_store.similarity_search(question, k=pool)]
    return rrf([lexical, dense])[:top]

ex = eval_set[0]
print("Q:", ex["question"], "\nA:", ex["answer"], "\nGold passages:", ex["supporting_titles"])
print("\nHybrid top-10:", retrieve(ex["question"]))

---
## 3. Stage 2: Rerank

The bi-encoder is fast but approximate, so we over-fetch 10 candidates and let a **cross-encoder** read each (question, passage) pair together. We keep 3.

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(question, candidate_titles, keep=3):
    scores = cross_encoder.predict([(question, text_of[t]) for t in candidate_titles])
    ranked = sorted(zip(candidate_titles, scores), key=lambda pair: pair[1], reverse=True)
    return [t for t, _ in ranked[:keep]]

print("Reranked top-3:", rerank(ex["question"], retrieve(ex["question"])))

---
## 4. Stage 3: Answer with Citations

The model gets the three passages **numbered** and must return JSON with its answer and the numbers of the passages it used. Citations are data, so we can check them against the gold supporting passages (the idea from `6.12`). If the passages do not contain the answer, it must say so.

In [ ]:
ANSWER_PROMPT = """Answer the question using ONLY the numbered passages below.
Reply with JSON only, in this exact form: {{"answer": "<short answer>", "sources": [<numbers of the passages you used>]}}
If the passages do not contain the answer, reply {{"answer": "I don't know", "sources": []}}.

{passages}

Question: {question}
JSON:"""

def parse_json(text):
    match = re.search(r"\{.*\}", text, re.S)
    try:
        data = json.loads(match.group(0)) if match else {}
    except json.JSONDecodeError:
        data = {}
    answer = str(data.get("answer", text)).strip()
    sources = [int(s) for s in data.get("sources", []) if str(s).isdigit()]
    return answer, sources

def answer_with_citations(question, context_titles):
    numbered = "\n\n".join(f"[{i}] {text_of[t]}" for i, t in enumerate(context_titles, start=1))
    reply = llm.invoke(ANSWER_PROMPT.format(passages=numbered, question=question)).content
    answer, sources = parse_json(reply)
    cited = [context_titles[s - 1] for s in sources if 1 <= s <= len(context_titles)]   # ignore invented numbers
    return answer, cited

def run_pipeline(question):
    candidates = retrieve(question)
    context = rerank(question, candidates)
    answer, cited = answer_with_citations(question, context)
    return {"candidates": candidates, "context": context, "answer": answer, "cited": cited}

out = run_pipeline(ex["question"])
print("Answer:", out["answer"], "| gold:", ex["answer"])
print("Cited :", out["cited"], "| gold passages:", ex["supporting_titles"])

---
## 5. Evaluate Each Stage

For every question we score three things separately:

| Stage | Metric |
|---|---|
| Retrieval | **both** gold passages are in the 3 passages the model saw (multi-hop needs both) |
| Citations | share of gold passages the model actually cited |
| Answer | token F1 and answer-in-context hit, as in the other notebooks |

In [ ]:
rows = []
for ex in eval_set:
    out = run_pipeline(ex["question"])
    gold = set(ex["supporting_titles"])
    docs = [Document(page_content=text_of[t]) for t in out["context"]]
    rows.append({
        **out, "question": ex["question"], "reference": ex["answer"], "gold": sorted(gold),
        "both_gold_in_context": int(gold <= set(out["context"])),
        "cite_recall": len(gold & set(out["cited"])) / len(gold) if gold else 1.0,
        "hit": rc.retrieval_hit(docs, ex["answer"]),
        "f1": rc.token_f1(out["answer"], ex["answer"]),
    })

print(f"{'Stage metric':<34}{'Score':>8}")
print("-" * 42)
print(f"{'Both gold passages in context':<34}{mean(r['both_gold_in_context'] for r in rows) * 100:>7.1f}%")
print(f"{'Gold passages cited (recall)':<34}{mean(r['cite_recall'] for r in rows) * 100:>7.1f}%")
print(f"{'Answer in context (hit)':<34}{mean(r['hit'] for r in rows) * 100:>7.1f}%")
print(f"{'Answer token F1':<34}{mean(r['f1'] for r in rows) * 100:>7.1f}%")

---
## 6. Which Stage Failed?

For each question with a wrong answer, the **first** stage that went wrong gives the label, so you know what to fix:

| Label | Meaning | Fix |
|---|---|---|
| `retrieval` | the answer is wrong and a gold passage never reached the model | chunking, hybrid weights, more candidates (6.6, 6.7) |
| `generation` | the answer is wrong although both gold passages were in the context | prompt, model, citation instructions |
| `ok` | answer matches (F1 of at least 0.5) | |

In [ ]:
def stage_of(r):
    if r["f1"] >= 0.5:
        return "ok"                      # the answer is right, nothing failed
    if not r["both_gold_in_context"]:
        return "retrieval"               # a gold passage never reached the model
    return "generation"                  # the evidence was there but the answer is wrong

labels = [stage_of(r) for r in rows]
for stage in ("ok", "retrieval", "generation"):
    print(f"{stage:<12}{labels.count(stage):>3} of {len(rows)}")

print("\nFailures:")
for r, label in zip(rows, labels):
    if label != "ok":
        print(f"[{label}] Q: {r['question'][:80]}\n   gold answer: {r['reference']!r} | model: {r['answer'][:60]!r}")
        print(f"   gold passages: {r['gold']} | context: {r['context']} | cited: {r['cited']}")

---
## 7. Add the Pipeline to the Ledger

The same sample, the same metrics, one more row. Compare the full pipeline with each single technique from `6.6` to `6.9`.

In [ ]:
rc.record_result("Capstone: hybrid → rerank → cite", "pipeline", rows=rows)
rc.show_ablation()

---
## Summary & Exercises

1. **A RAG pipeline is a chain of stages, and each stage can be measured on its own.** Retrieval quality, citation quality and answer quality tell you different things.
2. **Diagnose by first failing stage.** Tuning the prompt when the passage never reached the model is wasted effort.
3. **Compare techniques on one sample.** The ledger shows what each change bought in the same units.

**Exercises**
1. Swap hybrid retrieval for dense-only (`doc_store.similarity_search`) and rerun sections 5 to 7. Which stage metric moves?
2. Change `keep=3` to 2 and to 5 in `rerank`. How do retrieval recall and answer F1 trade off?
3. Make the model abstain: how many of the `generation` failures say "I don't know" versus guess? What would 6.12's abstention threshold do here?
4. Add `HyDE` from `6.8` in front of `retrieve`. Does the number of `retrieval` failures drop?